In [81]:
with open("data/input.txt", "r") as f:
    text = f.read()

In [82]:
print(len(text))

1115393


In [83]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [84]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(vocab_size)
print(''.join(chars))

65

 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [85]:
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[ch] for ch in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [86]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([1115393]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [87]:
n = int(0.9 * len(data))
train_data = data[:n]
test_data = data[n:]

In [88]:
block_size = 8
print(train_data[:block_size+1])

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])


In [89]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    print("when the input is", x[:t+1], ", target is", y[t])

when the input is tensor([18]) , target is tensor(47)
when the input is tensor([18, 47]) , target is tensor(56)
when the input is tensor([18, 47, 56]) , target is tensor(57)
when the input is tensor([18, 47, 56, 57]) , target is tensor(58)
when the input is tensor([18, 47, 56, 57, 58]) , target is tensor(1)
when the input is tensor([18, 47, 56, 57, 58,  1]) , target is tensor(15)
when the input is tensor([18, 47, 56, 57, 58,  1, 15]) , target is tensor(47)
when the input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) , target is tensor(58)


In [130]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == 'train' else test_data
    ix = torch.randint(len(data)-block_size-1, (batch_size,)) #randomly selects batch size number of starting tokens 
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    x = x.to(device)
    y = y.to(device)
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print("when the input is", context.tolist(), ", target is", target)

inputs:
torch.Size([4, 8])
tensor([[56, 41, 46,  1, 53,  5,  1, 58],
        [14, 33, 15, 23, 21, 26, 19, 20],
        [58, 46, 43, 56, 10,  0, 14, 59],
        [47, 60, 43,  1, 51, 43,  1, 58]], device='cuda:0')
targets:
torch.Size([4, 8])
tensor([[41, 46,  1, 53,  5,  1, 58, 46],
        [33, 15, 23, 21, 26, 19, 20, 13],
        [46, 43, 56, 10,  0, 14, 59, 58],
        [60, 43,  1, 51, 43,  1, 58, 46]], device='cuda:0')
----
when the input is [56] , target is tensor(41, device='cuda:0')
when the input is [56, 41] , target is tensor(46, device='cuda:0')
when the input is [56, 41, 46] , target is tensor(1, device='cuda:0')
when the input is [56, 41, 46, 1] , target is tensor(53, device='cuda:0')
when the input is [56, 41, 46, 1, 53] , target is tensor(5, device='cuda:0')
when the input is [56, 41, 46, 1, 53, 5] , target is tensor(1, device='cuda:0')
when the input is [56, 41, 46, 1, 53, 5, 1] , target is tensor(58, device='cuda:0')
when the input is [56, 41, 46, 1, 53, 5, 1, 58] , tar

In [91]:
n_embd = 64
n_head = 4

In [92]:
import torch
import torch.nn as nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd) #probability lookup table which is nn layer having 65*65 = 4225 trainable weights
        self.position_embedding_table = nn.Embedding(block_size, n_embd) #positional embeddings of block_size number of tokens. 8*64 trainable parameters

    def forward(self, idx, targets=None):

        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T)) #arange fun generate a 1D tensor ranging from 0 to T-1
        x = tok_emb + pos_emb # Pytorch broadcasting makes two different shaped vectors add together (B, T, C) + (T, C)

        self.lm_head = nn.Linear(n_embd, vocab_size)
        logits = self.lm_head(x)

        if targets == None:
            loss = None
        else:
            B, T, C = logits.shape #Batch, Time, Channels
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets) #Calculate Negative Log Likelihood

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))
# tokens
#   ↓
# token embeddings
#   +
# position embeddings
#   ↓
# self-attention
#   ↓
# contextual representations
#   ↓
# lm_head
#   ↓
# logits
#   ↓
# softmax
#   ↓
# probabilities
#   ↓
# sample next token
#   ↓
# append
#   ↓
# repeat

torch.Size([32, 65])
tensor(4.7115, grad_fn=<NllLossBackward0>)

EHBMoqxJjrqi?3MwHYTckkBKOzNszS-PxU'yXU;!XufpuB';j
yStwN-$uI!g-b
rwSyib&-WcNLh:TbtDBVl sa gAgpIf!.dfx


In [93]:
optimizer = torch.optim.AdamW(m.parameters(), lr = 1e-3)

In [94]:
batch_size = 32

for i in range(10000):
    xb, yb = get_batch("train")

    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True) #clears all prev gradients
    loss.backward() # compute gradients for all weights
    optimizer.step() # update the weights by using computed gradients  Wnew = Wold - lr * gradient
                     # Random batch -> Forward Pass(o/p prediction) -> Compute Loss -> Clear old gradients -> Backpropagation -> Update Weight -> repeat

print(loss.item())

4.173811912536621


In [95]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


RMEwwEKuyM;yooXL;quzDescFZ zUcOajyiiULTq$?$A.&3NTSLj:uuJvQUElXEiZCYsD lYzYf?m?bRfeSHhcSHc TR'VgFVJ3'sFTMrFq-wiPTMdjXhg'?zIrH,?dtrdOcnS.o$BVX lXbz$'gK vztSL&OjTIN;3&tq
Wya:$?.r&vD vDcNpOsp?vpHwJXUN?

:tirQAuB:yyKL$id$o;rlkulFtfGHfmbBNy:&AOn!XSK;w.CWkazDilX Nwe FfN
BVM!SFpR?-;TrIA;B'c:CQOl3SBa?ScR;
eQO CLo,YwFyELijtR PIX,ZntU$tvHPA!-CzNPzSh,!xQS-c,AeOIn:gVFJ,&WvhGANO,W:D.CtmciF
EG?XORqO$kGEmnkY&!CxLxn3
EqiQyzYQb3 i;JlJrKsUYEgjt.Qi!StzLsuHRdmcQyjV'Rwk:xuRs?UClfSJg.&eb
l
NunMshSI?ukBIV,YDKebSoZj?'V.


In [96]:
# toy example illustrating how matrix multiplication can be used for a "weighted aggregation"
# Self-attention is fundamentally a weighted averaging mechanism over token embeddings.
# Causal attention (Masked attention) restricts the model to only look at past and current tokens, preventing it from peeking to future tokens 
# using tril function(keeps only lower triangle)

torch.manual_seed(42)
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)

x.shape

torch.Size([4, 8, 2])

In [97]:
wei = torch.zeros(T, T)
wei = torch.masked_fill(wei, torch.tril(torch.ones(T, T)) == 0, float("-inf"))
wei = F.softmax(wei, dim=-1)
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [98]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [99]:
import torch

print(device)
print("PyTorch version:", torch.__version__)
print("PyTorch CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

cuda
PyTorch version: 2.13.0+cu130
PyTorch CUDA version: 13.0
CUDA available: True


In [100]:
out = wei @ x # Pytorch broadcasts the smaller tensor when doing (8, 8) x (4, 8, 2)
out # we are doing the weighted average of all previous and current token embeddings

tensor([[[ 1.9269,  1.4873],
         [ 1.4138, -0.3091],
         [ 1.1687, -0.6176],
         [ 0.8657, -0.8644],
         [ 0.5422, -0.3617],
         [ 0.3864, -0.5354],
         [ 0.2272, -0.5388],
         [ 0.1027, -0.3762]],

        [[ 1.6423, -0.1596],
         [ 0.5725,  0.1400],
         [ 0.1289,  0.4528],
         [ 0.2969,  0.7597],
         [ 0.4933,  0.8671],
         [ 0.5129,  0.9450],
         [ 0.4065,  0.8160],
         [ 0.3242,  0.8215]],

        [[-1.3847, -0.8712],
         [-0.8040,  0.4231],
         [-0.4297,  0.1405],
         [-0.2459, -0.0882],
         [-0.5082,  0.1285],
         [-0.5701,  0.0069],
         [-0.6707,  0.3092],
         [-0.7412,  0.2095]],

        [[-0.9138, -0.6581],
         [-0.4179, -0.0662],
         [-0.4413,  0.3530],
         [-0.5344,  0.0808],
         [-0.7082,  0.0718],
         [-0.6008,  0.1724],
         [-0.5289,  0.4113],
         [-0.6109,  0.5329]]])

In [101]:
# Currently, each token gives equal attention to previous tokens, but that's not optimal. We have to give individual attention weights
# So we use Q, K, V
# Query - What am I searching for?
# Key - What information do I have?
# Value - What information should I send if someone attends to me?

key = nn.Linear(C, C, bias=False)
query = nn.Linear(C, C, bias=False)
value = nn.Linear(C, C, bias=False)

k = key(x)
q = query(x)
v = value(x)

print(x.shape)
print(k.shape)
print(q.shape)
print(v.shape)

torch.Size([4, 8, 2])
torch.Size([4, 8, 2])
torch.Size([4, 8, 2])
torch.Size([4, 8, 2])


In [102]:
# scaled dot-product causal self-attention mechanism.
wei = q @ k.transpose(-2, -1) * C ** -0.5
print(wei.shape)
print(wei[0])

torch.Size([4, 8, 8])
tensor([[ 0.3627,  1.0575,  0.6851,  0.4901, -0.8484,  0.2751, -0.1378, -0.5746],
        [ 0.0988,  0.2581,  0.1678,  0.1167, -0.2072,  0.0632, -0.0375, -0.1420],
        [ 0.0833,  0.2240,  0.1455,  0.1020, -0.1798,  0.0558, -0.0316, -0.1228],
        [-0.0478, -0.1562, -0.1009, -0.0740,  0.1252, -0.0429,  0.0182,  0.0839],
        [-0.0852, -0.2247, -0.1461, -0.1018,  0.1804, -0.0553,  0.0324,  0.1235],
        [-0.1017, -0.3082, -0.1995, -0.1440,  0.2472, -0.0817,  0.0386,  0.1668],
        [-0.1370, -0.3993, -0.2587, -0.1850,  0.3203, -0.1039,  0.0520,  0.2169],
        [-0.1105, -0.3076, -0.1996, -0.1411,  0.2468, -0.0781,  0.0419,  0.1680]],
       grad_fn=<SelectBackward0>)


In [103]:
# Create Causal Mask (model should only aware of current and past token not on future tokens)
tril = torch.tril(torch.ones(T, T))

print(tril)

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])


In [104]:
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4603, 0.5397, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3110, 0.3580, 0.3310, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2618, 0.2349, 0.2483, 0.2550, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1961, 0.1706, 0.1846, 0.1929, 0.2558, 0.0000, 0.0000, 0.0000],
        [0.1635, 0.1330, 0.1483, 0.1567, 0.2318, 0.1668, 0.0000, 0.0000],
        [0.1346, 0.1035, 0.1192, 0.1283, 0.2126, 0.1391, 0.1626, 0.0000],
        [0.1156, 0.0949, 0.1057, 0.1121, 0.1652, 0.1194, 0.1346, 0.1527]],
       grad_fn=<SelectBackward0>)

In [105]:
out = wei @ v

In [106]:
# single head self attention
class Head(nn.Module): # Pytorch nn module

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size))) # Added causal mask

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        # calculate scaled attention scores
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)

        # applied causal mask - prevent tokens from looking into the future
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))

        # convert scores into attention weights
        wei = F.softmax(wei, dim=-1)

        # weighted aggregation of value vector
        out = wei @ v
        return out

In [107]:
head_size = 16
head = Head(head_size)

x_test = torch.randn(4, 8, 64)

out = head(x_test)
print(x_test.shape)
print(out.shape)

torch.Size([4, 8, 64])
torch.Size([4, 8, 16])


In [108]:
# multi-head attention
class MultiHeadAttention(nn.Module):
    """ Multiple heads of self attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()

        self.heads = nn.ModuleList(
            [Head(head_size) for _ in range(num_heads)]
        )

        self.proj = nn.Linear(n_embd, n_embd) # Linear projection to mix and transform all the o/p of each heads.
        dropout = 0.0
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out

In [109]:
num_heads = 4
head_size = 16

mha = MultiHeadAttention(num_heads, head_size)

x_test = torch.randn(4, 8, 64)

out = mha(x_test)
print(x_test.shape)
print(out.shape)

torch.Size([4, 8, 64])
torch.Size([4, 8, 64])


In [110]:
class FeedForward(nn.Module):
    """a simple linear layer followed by a non-linearity"""
    def __init__(self, n_embd):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout()
        )

    def forward(self, x):
        return self.net(x)

In [111]:
ffwd = FeedForward(n_embd)

x_test = torch.randn(4, 8, 64)

out = ffwd(x_test)
print("input: ", x_test.shape)
print("output: ", out.shape)

input:  torch.Size([4, 8, 64])
output:  torch.Size([4, 8, 64])


In [112]:
class Block(nn.Module):
    """Transformer block: communication followed by computation"""
    def __init__(self, n_embd, n_head):
        super().__init__()

        head_size = n_embd // n_head

        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)

        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x)) # Layer Norm -> Multi Head Attention -> Residual Connection -> Layer Norm -> ffwd nn -> Residual Connection
        x = x + self.ffwd(self.ln2(x))
        return x

In [113]:
block = Block(n_embd=64, n_head=4)

x_test = torch.randn(4, 8, 64)

out = block(x_test)

print("input: ", x_test.shape)
print("output: ", out.shape)

input:  torch.Size([4, 8, 64])
output:  torch.Size([4, 8, 64])


In [114]:
n_layer = 4

In [115]:
class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        self.blocks = nn.Sequential(
            *[Block(n_embd, n_head) for _ in range(n_layer)]
        )

        self.ln_f = nn.LayerNorm(n_embd)

        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))

        x = tok_emb + pos_emb

        x = self.blocks(x)

        x = self.ln_f(x)

        logits = self.lm_head(x)

        if targets == None:
            loss = None

        else:
            B, T, C = logits.shape

            logits = logits.view(B * T, C)
            targets = targets.view(B * T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):

        for _ in range(max_new_tokens):

            # Keep only the last block_size tokens as context
            idx_cond = idx[:, -block_size:]

            # Get predictions
            logits, loss = self(idx_cond)

            # We only care about the prediction for the last token
            logits = logits[:, -1, :]

            # Convert logits into probabilities
            probs = F.softmax(logits, dim=-1)

            # Sample the next token
            idx_next = torch.multinomial(probs, num_samples=1)

            # Append the new token to the sequence
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

# Architecture
#-------------------------------
# Characters
#     ↓
# Token IDs
#     ↓
# Token Embeddings
#     +
# Position Embeddings
#     ↓
# 4 Transformer Blocks
#     │
#     ├── Multi-Head Self-Attention
#     └── Feed-Forward Network
#     ↓
# Final LayerNorm
#     ↓
# Linear LM Head
#     ↓
# 65 logits
#     ↓
# Cross-Entropy Loss


In [147]:
model = GPTLanguageModel().to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3
)

print("Model device:", next(model.parameters()).device)
print("Parameters:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

Model device: cuda:0
Parameters: 0.208193 M


In [148]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)

generated = model.generate(context, max_new_tokens=200)

print(decode(generated[0].tolist()))


vBIfl&N!-LSvx:c&SHRpQXfsemUyGz,hR3DQtAjNDejV RWNQMEODaC'&LjGdw?ypmR$!GciJWaCzHzP&Qq3tZjr3L;nR3AxRxPSUsLmv&mPhAG'3s,OkeexpCRcZgyr$CROqT edS&ve;zfrU'hC.$YyueLqjLI?FREVqPB!fvzGaex,hPXOCPaGvLEK-sIxqXa'Erm


In [149]:
for iter in range(500):

    # Get random training batch
    xb, yb = get_batch("train")

    # Forward pass
    logits, loss = model(xb, yb)

    # Backpropagation
    optimizer.zero_grad(set_to_none=True)
    loss.backward()

    # Update weights
    optimizer.step()

    if iter % 100 == 0:
        print(f"step {iter}: loss = {loss.item():.4f}")


# ---- Immediately evaluate the SAME model ----

model.eval()

with torch.no_grad():

    xb, yb = get_batch("train")

    logits, train_loss = model(xb, yb)

print("\nAfter training:")
print("Train loss:", train_loss.item())

model.train()

step 0: loss = 4.3745
step 100: loss = 2.7446
step 200: loss = 2.4096
step 300: loss = 2.7105
step 400: loss = 2.4177

After training:
Train loss: 2.9100470542907715


GPTLanguageModel(
  (token_embedding_table): Embedding(65, 64)
  (position_embedding_table): Embedding(8, 64)
  (blocks): Sequential(
    (0): Block(
      (sa): MultiHeadAttention(
        (heads): ModuleList(
          (0-3): 4 x Head(
            (key): Linear(in_features=64, out_features=16, bias=False)
            (query): Linear(in_features=64, out_features=16, bias=False)
            (value): Linear(in_features=64, out_features=16, bias=False)
          )
        )
        (proj): Linear(in_features=64, out_features=64, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ffwd): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=64, out_features=256, bias=True)
          (1): ReLU()
          (2): Linear(in_features=256, out_features=64, bias=True)
          (3): Dropout(p=0.5, inplace=False)
        )
      )
      (ln1): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
      (ln2): LayerNorm((64,), eps=1e-05, elementw

In [150]:
eval_iters = 200

@torch.no_grad()
def estimate_loss():
    out = {}

    m.eval()

    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)

        for k in range(eval_iters):
            X, Y = get_batch(split)

            logits, loss = model(X, Y)

            losses[k] = loss.item()

        out[split] = losses.mean()

    m.train()

    return out

In [151]:
losses = estimate_loss()

print(f"train loss: {losses['train']:.4f}")
print(f"val loss:   {losses['val']:.4f}")

train loss: 2.6166
val loss:   2.5964


In [ ]:
# Text
#  ↓
# Character tokenizer
#  ↓
# Token embeddings
#  ↓
# Position embeddings
#  ↓
# Self-attention
#  ↓
# Multi-head attention
#  ↓
# Projection
#  ↓
# Feed-forward network
#  ↓
# Residual connections
#  ↓
# LayerNorm
#  ↓
# LM head
#  ↓
# Logits
#  ↓
# Cross-entropy loss
#  ↓
# Backpropagation
#  ↓
# AdamW
#  ↓
# Updated parameters